# 02 · Build an EOC common operating picture

**35 minutes · Intermediate.** Combine health reporting, exposure and clinic capacity while keeping their meanings separate. Construct a compact briefing panel.

[Run this lab in JupyterLite](https://jltobias.github.io/JupyterLite-World-Monitor-Prototype/lab/index.html?path=02_Cross_Domain_Monitor.ipynb) · [Book home](https://jltobias.github.io/JupyterLite-World-Monitor-Prototype/book/intro.html)

**Start:** run cells from top to bottom (Shift+Enter), or use Run → Run All Cells. The first launch downloads the Python runtime and packages. Core analysis uses bundled training data; internet is needed for initial setup and interactive JavaScript. Each lab starts independently.

In [ ]:
import sys
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "pandas", "matplotlib", "folium==0.20.0", "plotly==6.3.0"])
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from fieldguide import setup_style, load_tables, summarize, TRAINING
setup_style()
print(TRAINING)

## Start with a decision question
**Exercise question:** which districts need verification calls before the next planning meeting? The three panels below show reported burden, reporting completeness, and usable capacity. Their units remain separate. Correlated signals can motivate investigation without establishing a cause.

The original repository's optional Panel wrapper is preserved as `worldmonitor.panel_dashboard()`. This lab uses Matplotlib so every panel also appears in a static book and can be printed. For an optional live layout, install Panel with `await piplite.install("panel")` in Lite, then call the wrapper.

In [ ]:
s = summarize()
ordered = s.sort_values("reported_per_100k")
fig, axes = plt.subplots(1,3,figsize=(13,5),sharey=True)
for ax,metric,label,color in zip(axes,["reported_per_100k","completeness_pct","usable_beds"],
    ["Reports per 100,000 · 7 days","Reports received (%)","Usable available beds"],["#007f86","#c79522","#7662ad"]):
    ax.barh(ordered.district,ordered[metric],color=color)
    ax.set_xlabel(label); ax.grid(axis="x",alpha=.2)
axes[1].set_xlim(0,100)
fig.suptitle("SYNTHETIC · Common operating picture · 22–28 Nov 2025")
plt.tight_layout(); plt.show()

## Mark gaps before discussing priorities
Here 80% completeness is a **classroom review trigger**, not a WHO surveillance standard. A flag requests verification. It does not impute the missing cases or rule out urgency in an unflagged district.

In [ ]:
review = s.loc[(s.completeness_pct<80) | (s.functional==0),
               ["district","completeness_pct","functional","access_delay_hours"]]
display(review.round(1))
print("Observed cases:", int(s.cases.sum()))
print("Reporting completeness:", round(100*s.received.sum()/s.expected.sum(),1), "%")

In [ ]:
from worldmonitor import response_body
# Preserve the cross-domain API exploration: count records, never combine
# country-risk, money and health into an unexplained index.
for operation in ["GetCountryRisk","ListMarketQuotes","GetChokepointStatus","GetForecasts"]:
    body = response_body(operation)
    print(operation, "sample fields:", ", ".join(body))

## Handover structure
![Observe, verify, analyze, brief and review cycle](assets/workflow.svg)
Use a dated snapshot at handover. Link each panel to its source register. Assign a person to resolve each gap and record when the next update is due. Book readers see a reproducible exercise; Lite readers can edit the thresholds and see what changes.

## Practice and debrief

Create a three-sentence handover: what changed, what is uncertain, and what you need verified by noon. Explain why a country-level sandbox risk score cannot serve as a district disease-incidence estimate.

## Sources and further learning

[WHO EOC framework](https://www.who.int/publications/i/item/framework-for-a-public-health-emergency-operations-centre) · [World Monitor architecture](https://www.worldmonitor.app/docs/architecture).